# Semana 14 — Despliegue en Cloud

**Objetivo:** llevar la DocBot API de tu máquina a una URL pública en Railway o Render.

Este notebook es diferente a los anteriores: en lugar de ejecutar código Python interactivo,
genera los archivos que necesitas y te guía paso a paso por el proceso de despliegue.

**Al finalizar tendrás:**
- Un repositorio GitHub listo para CI/CD
- Un pipeline de GitHub Actions que ejecuta tests antes de desplegar
- La DocBot API desplegada en Railway o Render con URL pública
- Un script de smoke test que verifica el despliegue

**Prerrequisitos:**
- `proyectos/docbot-api/` completo (semana 13)
- Cuenta en [Railway](https://railway.app) o [Render](https://render.com) (ambas gratuitas)
- Cuenta en GitHub
- Railway CLI instalado: `npm install -g @railway/cli` (opcional, para deploy desde terminal)

## Celda 1 — Preparar el repositorio

Primero verifica que el proyecto está en orden antes de subirlo a GitHub.

In [ ]:
import os
import subprocess
from pathlib import Path

# Ruta al proyecto docbot-api
PROJECT_DIR = Path("../proyectos/docbot-api").resolve()
print(f"Proyecto: {PROJECT_DIR}")
print(f"Existe: {PROJECT_DIR.exists()}")

# Archivos que deben existir
ARCHIVOS_REQUERIDOS = [
    "api.py",
    "models.py",
    "Dockerfile",
    ".dockerignore",
    "requirements.txt",
    "test_api.py",
]

print("\nChecklist de archivos:")
todos_ok = True
for archivo in ARCHIVOS_REQUERIDOS:
    ruta = PROJECT_DIR / archivo
    estado = "✓" if ruta.exists() else "✗ FALTA"
    print(f"  {estado}  {archivo}")
    if not ruta.exists():
        todos_ok = False

if todos_ok:
    print("\n✓ Proyecto listo para despliegue")
else:
    print("\n✗ Faltan archivos — ejecuta primero el notebook de la semana 13")

## Celda 2 — Generar .gitignore

El `.gitignore` impide que secretos y datos locales lleguen a GitHub.

In [ ]:
GITIGNORE = """# Secretos
.env
*.env

# Datos persistentes (van en volúmenes cloud, no en el repo)
chroma_db/
sesiones/
evaluaciones/

# Python
__pycache__/
*.pyc
*.pyo
.venv/
venv/
*.egg-info/
dist/
build/

# Jupyter
.ipynb_checkpoints/

# IDE
.vscode/
.idea/
*.swp

# macOS
.DS_Store
"""

ruta_gitignore = PROJECT_DIR / ".gitignore"
ruta_gitignore.write_text(GITIGNORE)
print(f"✓ .gitignore generado en {ruta_gitignore}")

# Verificar que .env no está en el proyecto (si existe, no debe subirse)
env_file = PROJECT_DIR / ".env"
if env_file.exists():
    print("⚠️  Existe un .env — está en .gitignore, no se subirá a GitHub")
else:
    print("✓ No hay .env en el proyecto (correcto)")

## Celda 3 — Pipeline de GitHub Actions

Este pipeline se ejecuta en cada `git push` a `main`:
1. Ejecuta los tests
2. Si pasan, despliega en Railway

**Nota:** el step de Railway requiere un token. En el paso de configuración de Railway verás cómo obtenerlo.

In [ ]:
import os

WORKFLOW_DIR = PROJECT_DIR / ".github" / "workflows"
WORKFLOW_DIR.mkdir(parents=True, exist_ok=True)

DEPLOY_YML = """name: Test and Deploy

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install dependencies
        run: pip install -r requirements.txt

      - name: Run tests
        run: pytest test_api.py -v --tb=short
        env:
          # Los tests de integración con LLM necesitan la key real.
          # Define OPENAI_API_KEY en GitHub → Settings → Secrets.
          # Si tus tests mockean el LLM, puedes usar un valor falso aquí.
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}

  deploy:
    needs: test
    runs-on: ubuntu-latest
    # Solo despliega en push a main, no en PRs
    if: github.event_name == 'push' && github.ref == 'refs/heads/main'

    steps:
      - uses: actions/checkout@v4

      - name: Install Railway CLI
        run: npm install -g @railway/cli

      - name: Deploy to Railway
        run: railway up --detach
        env:
          RAILWAY_TOKEN: ${{ secrets.RAILWAY_TOKEN }}
"""

ruta_workflow = WORKFLOW_DIR / "deploy.yml"
ruta_workflow.write_text(DEPLOY_YML)
print(f"✓ Pipeline generado: {ruta_workflow}")
print()
print("Próximos pasos:")
print("  1. Crea el repositorio en GitHub")
print("  2. Añade OPENAI_API_KEY y RAILWAY_TOKEN en GitHub → Settings → Secrets")
print("  3. git push origin main → el pipeline se activa automáticamente")

## Celda 4 — Healthcheck mejorado en el Dockerfile

El Dockerfile de la semana 13 expone el puerto pero no tiene HEALTHCHECK.
Lo añadimos para que Railway/Render sepa cuándo el contenedor está listo.

In [ ]:
DOCKERFILE = """FROM python:3.11-slim

WORKDIR /app

# Instala dependencias primero — se cachea si requirements.txt no cambia
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copia el código
COPY . .

# Crea directorios para volúmenes
RUN mkdir -p chroma_db sesiones evaluaciones

EXPOSE 8000

# Railway/Render sondea este endpoint para saber si el contenedor está vivo.
# --interval=30s: cada 30 segundos
# --timeout=10s:  si no responde en 10s, cuenta como fallo
# --start-period=15s: espera 15s antes del primer check (tiempo de arranque)
# --retries=3: 3 fallos consecutivos → el contenedor se marca como unhealthy
HEALTHCHECK --interval=30s --timeout=10s --start-period=15s --retries=3 \\
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "api:app", "--host", "0.0.0.0", "--port", "8000"]
"""

ruta_dockerfile = PROJECT_DIR / "Dockerfile"
ruta_dockerfile.write_text(DOCKERFILE)
print(f"✓ Dockerfile actualizado con HEALTHCHECK: {ruta_dockerfile}")

# Muestra el Dockerfile
print()
print(ruta_dockerfile.read_text())

## Celda 5 — Script de smoke test

Un smoke test verifica que la API desplegada responde correctamente.
Lo ejecutas después de cada despliegue con la URL real de producción.

In [ ]:
SMOKE_TEST = '''#!/usr/bin/env python3
"""
Smoke test para la DocBot API en producción.
Uso: python smoke_test.py https://tu-api.railway.app
"""
import sys
import time
import json
import urllib.request
import urllib.error

def get(url: str, timeout: int = 10) -> dict:
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return json.loads(r.read())

def post(url: str, data: dict, timeout: int = 30) -> dict:
    body = json.dumps(data).encode()
    req = urllib.request.Request(
        url, data=body,
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())

def test(nombre: str, fn):
    try:
        inicio = time.time()
        resultado = fn()
        latencia = (time.time() - inicio) * 1000
        print(f"  ✓  {nombre} ({latencia:.0f}ms)")
        return resultado
    except Exception as e:
        print(f"  ✗  {nombre}: {e}")
        return None

def main():
    if len(sys.argv) < 2:
        print("Uso: python smoke_test.py https://tu-api.railway.app")
        sys.exit(1)

    base = sys.argv[1].rstrip("/")
    print(f"\nSmoke test → {base}\n")

    fallos = 0

    # 1. Health check
    r = test("GET /health", lambda: get(f"{base}/health"))
    if r is None or r.get("status") != "ok":
        print("    → La API no responde o status != ok")
        fallos += 1

    # 2. Ingesta de un artículo de ejemplo
    r = test("POST /ingest (Wikipedia: FastAPI)", lambda: post(
        f"{base}/ingest",
        {"fuentes": [{"tipo": "wikipedia", "valor": "FastAPI",
                       "titulo": "FastAPI", "categoria": "test"}]}
    ))
    if r is None or not r.get("ok"):
        print("    → Fallo en ingesta")
        fallos += 1

    # 3. Pregunta sobre el corpus
    r = test("POST /chat (pregunta sobre corpus)", lambda: post(
        f"{base}/chat",
        {"pregunta": "What is FastAPI?", "user_id": "smoke_test"}
    ))
    if r is None or not r.get("respuesta"):
        print("    → Fallo en chat")
        fallos += 1
    elif r:
        print(f"    → respondida={r.get('respondida')} latencia={r.get('latencia_ms', 0):.0f}ms")

    # 4. Validación: request inválido debe dar 422
    try:
        post(f"{base}/chat", {"pregunta": "", "user_id": "smoke_test"})
        print("  ✗  POST /chat (pregunta vacía) → debería dar 422 pero no lo hizo")
        fallos += 1
    except urllib.error.HTTPError as e:
        if e.code == 422:
            print("  ✓  POST /chat (pregunta vacía) → 422 correcto")
        else:
            print(f"  ✗  POST /chat (pregunta vacía) → código inesperado {e.code}")
            fallos += 1

    # Resultado
    print()
    if fallos == 0:
        print("✓ Todos los tests pasaron — la API está operativa")
    else:
        print(f"✗ {fallos} test(s) fallaron")
        sys.exit(1)

if __name__ == "__main__":
    main()
'''

ruta_smoke = PROJECT_DIR / "smoke_test.py"
ruta_smoke.write_text(SMOKE_TEST)
print(f"✓ smoke_test.py generado en {ruta_smoke}")
print()
print("Uso tras el despliegue:")
print("  python smoke_test.py https://tu-api.railway.app")

## Celda 6 — Guía paso a paso: despliegue en Railway

Ejecuta esta celda para ver las instrucciones completas de despliegue.

In [ ]:
GUIA = """
============================================================
 GUÍA DE DESPLIEGUE — Railway
============================================================

PASO 1 — Crear el repositorio en GitHub
---------------------------------------
  cd proyectos/docbot-api
  git init
  git add .
  git commit -m "DocBot API v1.0"
  # Crea el repo en github.com/new, luego:
  git remote add origin https://github.com/TU_USUARIO/docbot-api.git
  git push -u origin main

PASO 2 — Crear el proyecto en Railway
--------------------------------------
  1. Ve a railway.app y haz login con GitHub
  2. New Project → Deploy from GitHub repo → selecciona docbot-api
  3. Railway detecta el Dockerfile automáticamente

PASO 3 — Configurar variables de entorno
-----------------------------------------
  En Railway → tu servicio → Variables → Add Variable:
    OPENAI_API_KEY = sk-...
    API_KEYS       = clave1,clave2   (opcional, si activaste auth)
    LOG_LEVEL      = INFO

PASO 4 — Obtener el token de Railway para GitHub Actions
---------------------------------------------------------
  railway.app → Account Settings → Tokens → New Token
  Nombre: github-actions
  Copia el token.

  En GitHub → tu repo → Settings → Secrets → New secret:
    RAILWAY_TOKEN  = (pega el token)
    OPENAI_API_KEY = sk-...

PASO 5 — Primer despliegue
---------------------------
  Railway ya está construyendo la imagen desde el push inicial.
  Espera ~2 minutos y Railway te da una URL pública:
    https://docbot-api-production.railway.app

PASO 6 — Verificar con el smoke test
-------------------------------------
  python smoke_test.py https://docbot-api-production.railway.app

PASO 7 — CI/CD en acción
-------------------------
  Modifica cualquier archivo y haz git push.
  GitHub Actions → ejecuta tests → si pasan → Railway despliega.
  Puedes ver el pipeline en GitHub → Actions tab.

============================================================
 ALTERNATIVA: Render (free tier)
============================================================

  1. render.com → New → Web Service → Connect GitHub repo
  2. Environment: Docker
  3. Añade las variables de entorno en el panel
  4. Deploy → espera ~3 minutos

  ⚠️  El free tier de Render duerme la instancia tras 15min de inactividad.
  El primer request tras el sleep tarda ~30s.
  Para demos, usa Railway (no duerme con el plan Hobby a $5/mes).
"""

print(GUIA)

## Celda 7 — Verificar el despliegue en vivo

Una vez desplegada la API, ejecuta esta celda con tu URL real para verificar que todo funciona.

In [ ]:
import urllib.request
import urllib.error
import json
import time

# TODO: sustituye con tu URL real de Railway o Render
API_URL = "https://TU-API.railway.app"

# Si aún no tienes la URL, esta celda muestra un aviso
if "TU-API" in API_URL:
    print("⚠️  Sustituye API_URL con la URL real de tu despliegue")
    print("    Ejemplo: API_URL = 'https://docbot-api-production.railway.app'")
else:
    print(f"Verificando {API_URL}...\n")

    try:
        inicio = time.time()
        with urllib.request.urlopen(f"{API_URL}/health", timeout=15) as r:
            data = json.loads(r.read())
            latencia_ms = (time.time() - inicio) * 1000

        print(f"✓ /health → {data}")
        print(f"  Latencia: {latencia_ms:.0f}ms")

        if latencia_ms > 2000:
            print("  ⚠️  Latencia alta — posiblemente la instancia estaba durmiendo (Render free tier)")
        else:
            print("  → La API responde rápido, instancia activa")

    except urllib.error.URLError as e:
        print(f"✗ No se puede conectar: {e}")
        print("  Comprueba que la URL es correcta y el servicio está desplegado")

## Celda 8 — Monitor de costes y latencia

Un monitor básico que llama al endpoint `/metrics/{user_id}` y muestra un resumen.
Útil para hacer seguimiento del coste durante demostraciones a clientes.

In [ ]:
import urllib.request
import json

# TODO: sustituye con tu URL y user_id real
API_URL  = "https://TU-API.railway.app"
USER_ID  = "test_user"

if "TU-API" in API_URL:
    # Simulación con datos de ejemplo para ver el formato
    print("[Modo simulación — sustituye API_URL con la URL real]\n")
    metricas = {
        "user_id": "test_user",
        "n_preguntas": 12,
        "n_respondidas": 10,
        "latencia_media_ms": 1843.2,
        "faithfulness_media": 0.87,
    }
else:
    with urllib.request.urlopen(f"{API_URL}/metrics/{USER_ID}", timeout=10) as r:
        metricas = json.loads(r.read())

# Muestra el resumen
print("=" * 40)
print(f" Métricas — usuario: {metricas.get('user_id', USER_ID)}")
print("=" * 40)

n_p = metricas.get("n_preguntas", 0)
n_r = metricas.get("n_respondidas", 0)
tasa = (n_r / n_p * 100) if n_p > 0 else 0

print(f"  Preguntas totales:   {n_p}")
print(f"  Respondidas:         {n_r} ({tasa:.0f}%)")
print(f"  Latencia media:      {metricas.get('latencia_media_ms', 0):.0f}ms")
print(f"  Faithfulness media:  {metricas.get('faithfulness_media', 0):.2f}")

# Estimación de coste con gpt-4o-mini
# ~500 tokens por pregunta (prompt + respuesta) a $0.15/1M input + $0.60/1M output
coste_estimado = n_p * 500 * (0.15 + 0.60) / 2 / 1_000_000
print(f"  Coste estimado:      ${coste_estimado:.4f}")
print()

# Alertas
if tasa < 70:
    print("⚠️  Tasa de respuesta baja (<70%) — considera ampliar el corpus")
if metricas.get("latencia_media_ms", 0) > 3000:
    print("⚠️  Latencia alta (>3s) — revisa los logs en Railway/Render")
if metricas.get("faithfulness_media", 1) < 0.7:
    print("⚠️  Faithfulness baja (<0.7) — revisa el prompt y los chunks recuperados")

## Reto — Añadir un endpoint `/status` con info del despliegue

El health check actual solo dice `{"status": "ok"}`. Para producción es útil tener más información:
versión, tiempo de arranque, número de usuarios activos.

**Implementa** el endpoint `GET /status` en `api.py` que devuelva:

```json
{
  "status": "ok",
  "version": "1.0.0",
  "uptime_seconds": 3842,
  "usuarios_activos": 3,
  "modelo": "gpt-4o-mini"
}
```

Pistas:
- Guarda `_start_time = time.time()` en el lifespan al arrancar
- `uptime_seconds = int(time.time() - _start_time)`
- `usuarios_activos = len(_bots)` (número de bots en el pool)
- Añade el modelo como constante en `api.py`

Una vez implementado, verifica que el smoke test lo detecta añadiendo un caso:
```python
r = test("GET /status", lambda: get(f"{base}/status"))
assert r["uptime_seconds"] >= 0
```

In [ ]:
# Esqueleto para el reto
# Implementa el endpoint /status en api.py

SKELETON_STATUS = '''
# En api.py — añade esto:

import time

MODELO   = "gpt-4o-mini"
VERSION  = "1.0.0"
_start_time: float = 0.0

@asynccontextmanager
async def lifespan(app: FastAPI):
    global _start_time
    _start_time = time.time()   # <-- añade esta línea al lifespan existente
    # ... resto del lifespan ...
    yield
    # ... shutdown ...


class StatusResponse(BaseModel):
    status:           str
    version:          str
    uptime_seconds:   int
    usuarios_activos: int
    modelo:           str


@app.get("/status", response_model=StatusResponse)
async def status() -> StatusResponse:
    # TODO: implementa el cuerpo del endpoint
    pass
'''

print(SKELETON_STATUS)

## Resumen de la semana 14

| Concepto | Dónde lo aplicaste |
|---------|-------------------|
| `.gitignore` con secretos y datos | Celda 2 |
| Pipeline CI/CD con GitHub Actions | Celda 3 |
| HEALTHCHECK en Dockerfile | Celda 4 |
| Smoke test para producción | Celda 5 |
| Guía de despliegue Railway/Render | Celda 6 |
| Verificación del despliegue en vivo | Celda 7 |
| Monitor de costes y latencia | Celda 8 |
| Endpoint `/status` extendido | Reto |

**Bloque 6 completado.** DocBot ya tiene URL pública, tests automáticos y despliegue continuo.

La semana 15 arranca el Bloque 7: **Proyecto 2 — Agente autónomo** para un caso de uso real de negocio.